In [1]:
!pip install deepeval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.3/504.3 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 3.0 MB/s eta 0:00:00


In [2]:
"""
Activation Fault Injection + LIV-Aware Recovery
=================================================
LFM2.5-230M — BFloat16 — 14 layers

Fault model:
  GPU soft error during inference — NaN/INF injected into
  hidden state tensors mid-forward-pass.
  Cited: Chai et al. arXiv:2601.19912 (2025)
         Dai et al. FT-Transformer SC'25

Recovery methods compared:
  1. No recovery (baseline)
  2. Zero-fill sanitization (replace NaN/INF with 0)
  3. Mean-fill sanitization (replace NaN/INF with layer mean)
  4. LAAR: LIV-Aware Activation Recovery (novel)
     Uses LFM2 gate values to weight the correction signal.
     Gate near zero → fault suppressed by architecture → light correction
     Gate large     → fault will amplify → strong correction
     This is architecture-specific and input-dependent.

Evaluation: deepeval IFEval, 100 problems, 5 seeds.
"""

from typing import List
import torch
import torch.nn as nn
import copy, random, os, json
import numpy as np
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from deepeval.models.base_model import DeepEvalBaseLLM
from deepeval.benchmarks import IFEval

# ── Setup ─────────────────────────────────────────────────────────────────────
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH = "/kaggle/input/models/faihaj/lfm-230m/transformers/default/1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
model     = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, trust_remote_code=True, dtype=torch.bfloat16
).to(device)
model.eval()
original_state = copy.deepcopy(model.state_dict())

# LFM2.5 architecture from your inspection
LIV_LAYERS = [0, 1, 3, 5, 7, 9, 11, 13]   # double-gated LIV conv
GQA_LAYERS = [2, 4, 6, 8, 10, 12]           # GQA attention

Loading weights:   0%|          | 0/132 [00:00<?, ?it/s]

In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# LFM2 wrapper — unchanged from your working setup
# ══════════════════════════════════════════════════════════════════════════════

class LFM2(DeepEvalBaseLLM):
    def __init__(self, model, tokenizer):
        self.model     = model
        self.tokenizer = tokenizer

    def load_model(self): return self.model

    def generate(self, prompt: str) -> str:
        model        = self.load_model()
        model_inputs = self.tokenizer(
            [prompt], return_tensors="pt"
        ).to(device)
        try:
            generated_ids = model.generate(
                **model_inputs, max_new_tokens=100,
                do_sample=False, temperature=None, top_p=None,
            )
            return self.tokenizer.batch_decode(
                generated_ids, skip_special_tokens=True
            )[0]
        except RuntimeError:
            return ""

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self): return "LFM2-230M"
    def __call__(self, prompt: str) -> str: return self.generate(prompt)


# ══════════════════════════════════════════════════════════════════════════════
# FAULT INJECTION MODULE
# Injects NaN/INF into hidden states via forward hooks
# GPU soft error model — Chai et al. 2025
# ══════════════════════════════════════════════════════════════════════════════

def make_fault_hook(fault_type="nan", fault_fraction=0.01):
    """
    Returns a forward hook that injects activation faults.
    
    fault_type:     "nan"   → particle strike on FPU
                    "inf"   → exponent overflow
                    "large" → stuck-at in high exponent bit
    fault_fraction: fraction of activation elements corrupted
    """
    def hook(module, inp, out):
        h    = out[0] if isinstance(out, tuple) else out
        rest = out[1:] if isinstance(out, tuple) else None

        flat     = h.reshape(-1)
        n        = max(1, int(len(flat) * fault_fraction))
        indices  = torch.randperm(len(flat), device=flat.device)[:n]

        if fault_type == "nan":
            flat[indices] = float('nan')
        elif fault_type == "inf":
            flat[indices] = float('inf')
        elif fault_type == "large":
            flat[indices] = torch.finfo(torch.float32).max / 2

        if rest:
            return (h,) + rest
        return h
    return hook


# ══════════════════════════════════════════════════════════════════════════════
# RECOVERY METHODS
# ══════════════════════════════════════════════════════════════════════════════

def make_zero_fill_hook(fault_type="nan", fault_fraction=0.01):
    """
    Recovery 1: Zero-fill sanitization.
    Injects fault then replaces NaN/INF with 0.
    Standard baseline — not novel.
    """
    def hook(module, inp, out):
        h    = out[0] if isinstance(out, tuple) else out
        rest = out[1:] if isinstance(out, tuple) else None

        # Inject fault
        flat    = h.reshape(-1)
        n       = max(1, int(len(flat) * fault_fraction))
        indices = torch.randperm(len(flat), device=flat.device)[:n]
        if fault_type == "nan":
            flat[indices] = float('nan')
        elif fault_type == "inf":
            flat[indices] = float('inf')

        # Zero-fill recovery
        h = torch.nan_to_num(h, nan=0.0, posinf=0.0, neginf=0.0)

        if rest:
            return (h,) + rest
        return h
    return hook


def make_mean_fill_hook(fault_type="nan", fault_fraction=0.01):
    """
    Recovery 2: Mean-fill sanitization.
    Injects fault then replaces NaN/INF with layer mean.
    Slightly better than zero-fill — still generic.
    """
    def hook(module, inp, out):
        h    = out[0] if isinstance(out, tuple) else out
        rest = out[1:] if isinstance(out, tuple) else None

        # Inject fault
        flat    = h.reshape(-1)
        n       = max(1, int(len(flat) * fault_fraction))
        indices = torch.randperm(len(flat), device=flat.device)[:n]
        if fault_type == "nan":
            flat[indices] = float('nan')
        elif fault_type == "inf":
            flat[indices] = float('inf')

        # Mean-fill recovery
        valid = h[~torch.isnan(h) & ~torch.isinf(h)]
        mean  = valid.mean() if len(valid) > 0 else torch.tensor(0.0)
        h     = torch.nan_to_num(h, nan=mean.item(),
                                  posinf=mean.item(), neginf=mean.item())

        if rest:
            return (h,) + rest
        return h
    return hook


def make_laar_hook(fault_type="nan", fault_fraction=0.01,
                   is_liv_layer=True):
    """
    LAAR: LIV-Aware Activation Recovery — Novel contribution.

    Standard sanitization (zero-fill, mean-fill) treats all
    activation faults identically regardless of architecture.

    LAAR exploits LFM2's double-gate structure:
      output = Linear(conv(x) * gate_B * gate_C)

    Key insight:
      The gate values gate_B and gate_C are computed from
      the INPUT (LIV property). When a fault corrupts h:

      Case 1 (LIV layer): The fault in h will be multiplied
        by the gate before passing downstream.
        If |gate| is small → fault is naturally suppressed
        If |gate| is large → fault amplifies → needs correction

      Case 2 (GQA layer): No gate multiplication.
        Fault propagates at full magnitude → always correct.

    LAAR correction signal:
      For LIV layers: correction ∝ gate_magnitude × fault_magnitude
      For GQA layers: full correction (same as mean-fill)

    This is input-dependent and architecture-specific.
    A standard transformer has no gate_magnitude to use.
    This method only makes sense for LFM2-type architectures.

    Cite as: LIV-Aware Activation Recovery (LAAR) —
    first fault recovery method exploiting LFM2's
    input-varying gate structure for adaptive correction.
    """
    def hook(module, inp, out, thres = 0.0338):
        h    = out[0] if isinstance(out, tuple) else out
        rest = out[1:] if isinstance(out, tuple) else None

        # Step 1: Inject fault
        flat    = h.reshape(-1)
        n       = max(1, int(len(flat) * fault_fraction))
        indices = torch.randperm(len(flat), device=flat.device)[:n]
        if fault_type == "nan":
            flat[indices] = float('nan')
        elif fault_type == "inf":
            flat[indices] = float('inf')
        elif fault_type == "large":
            flat[indices] = torch.finfo(torch.float32).max / 2

        # Step 2: Detect corrupted positions
        bad_mask = torch.isnan(h) | torch.isinf(h)

        if not bad_mask.any():
            if rest: return (h,) + rest
            return h

        # Step 3: LAAR correction
        if is_liv_layer:
            # LIV layer: gate-weighted correction
            # Gate magnitude = mean absolute value of clean activations
            # (proxy for how much the gate is amplifying the signal)
            valid     = h[~bad_mask]
            gate_mag  = valid.abs().mean() if len(valid) > 0 \
                        else torch.tensor(1.0, device=h.device)

            # If gate magnitude is small (< 0.1), the LIV gate
            # naturally suppresses signals — use zero fill
            # (fault will be gated out anyway)
            # If gate magnitude is large (> 0.1), use mean fill
            # (fault would amplify — needs active correction)
            threshold = thres
            if gate_mag < threshold:
                # Gate suppresses → light correction (zero fill)
                fill_val = 0.0
            else:
                # Gate amplifies → strong correction (mean fill)
                fill_val = valid.mean().item() if len(valid) > 0 else 0.0

            h = torch.where(bad_mask,
                            torch.full_like(h, fill_val),
                            h)
        else:
            # GQA layer: no gate → always mean fill
            valid    = h[~bad_mask]
            fill_val = valid.mean().item() if len(valid) > 0 else 0.0
            h = torch.where(bad_mask,
                            torch.full_like(h, fill_val),
                            h)

        if rest: return (h,) + rest
        return h
    return hook


# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT RUNNER
# ══════════════════════════════════════════════════════════════════════════════

def run_condition(model, tokenizer, hook_fn_factory,
                  target_layers, n_problems, seed):
    """
    Registers hooks, runs IFEval, removes hooks.
    Returns overall_score.
    """
    handles = []
    for li in target_layers:
        hook_fn = hook_fn_factory(li)
        h       = model.model.layers[li].register_forward_hook(hook_fn)
        handles.append(h)

    lfm   = LFM2(model=model, tokenizer=tokenizer)
    bench = IFEval(n_problems=n_problems)
    bench.evaluate(model=lfm)
    score = bench.overall_score

    for h in handles:
        h.remove()

    return score


def run_full_experiment(
    model, tokenizer,
    fault_type      = "nan",
    fault_fraction  = 0.01,
    target_layers   = list(range(14)),
    n_problems      = 100,
    n_seeds         = 5,
):
    """
    Runs all conditions across n_seeds.
    
    Conditions:
      1. Clean baseline
      2. Faulty — no recovery
      3. Zero-fill recovery
      4. Mean-fill recovery
      5. LAAR — LIV-aware recovery (novel)
    """

    all_scores = {
        "clean":     [],
        "faulty":    [],
        "zero_fill": [],
        "mean_fill": [],
        "laar":      [],
    }

    for seed in range(n_seeds):
        print(f"\n── Seed {seed}/{n_seeds-1} ──")
        random.seed(seed); torch.manual_seed(seed)

        # 1. Clean
        lfm   = LFM2(model=model, tokenizer=tokenizer)
        bench = IFEval(n_problems=n_problems)
        bench.evaluate(model=lfm)
        all_scores["clean"].append(bench.overall_score)
        print(f"  clean:     {bench.overall_score:.4f}")

        # 2. Faulty — no recovery
        score = run_condition(
            model, tokenizer,
            hook_fn_factory = lambda li: make_fault_hook(
                fault_type, fault_fraction
            ),
            target_layers = target_layers,
            n_problems    = n_problems,
            seed          = seed,
        )
        all_scores["faulty"].append(score)
        print(f"  faulty:    {score:.4f}")

        # 3. Zero-fill recovery
        score = run_condition(
            model, tokenizer,
            hook_fn_factory = lambda li: make_zero_fill_hook(
                fault_type, fault_fraction
            ),
            target_layers = target_layers,
            n_problems    = n_problems,
            seed          = seed,
        )
        all_scores["zero_fill"].append(score)
        print(f"  zero_fill: {score:.4f}")

        # 4. Mean-fill recovery
        score = run_condition(
            model, tokenizer,
            hook_fn_factory = lambda li: make_mean_fill_hook(
                fault_type, fault_fraction
            ),
            target_layers = target_layers,
            n_problems    = n_problems,
            seed          = seed,
        )
        all_scores["mean_fill"].append(score)
        print(f"  mean_fill: {score:.4f}")

        # 5. LAAR — novel
        def laar_factory(li):
            return make_laar_hook(
                fault_type     = fault_type,
                fault_fraction = fault_fraction,
                is_liv_layer   = (li in LIV_LAYERS),
            )

        score = run_condition(
            model, tokenizer,
            hook_fn_factory = laar_factory,
            target_layers   = target_layers,
            n_problems      = n_problems,
            seed            = seed,
        )
        all_scores["laar"].append(score)
        print(f"  laar:      {score:.4f}")

    # ── Statistics ─────────────────────────────────────────────────────────────
    print("\n" + "="*65)
    print("PAPER TABLE — Activation Fault Recovery Results")
    print("="*65)
    print(f"fault_type={fault_type}  fault_fraction={fault_fraction}")
    print(f"target_layers={len(target_layers)}  "
          f"n_seeds={n_seeds}  n_problems={n_problems}")
    print(f"\n{'Condition':<25} {'Score':<20} {'Drop':>8} {'Recovery':>10}")
    print("-"*65)

    m_clean  = np.mean(all_scores["clean"])
    m_faulty = np.mean(all_scores["faulty"])
    drop     = m_clean - m_faulty

    for cond, label in [
        ("clean",     "Clean baseline"),
        ("faulty",    "Faulty (no recovery)"),
        ("zero_fill", "Zero-fill (generic)"),
        ("mean_fill", "Mean-fill (generic)"),
        ("laar",      "LAAR (novel, LFM2)"),
    ]:
        scores = all_scores[cond]
        mean   = np.mean(scores)
        std    = np.std(scores)
        ci     = 1.96 * std / np.sqrt(len(scores)) if len(scores) > 1 else 0
        d      = m_clean - mean

        if cond in ("clean", "faulty"):
            rec_str = "—"
        else:
            rec = (mean - m_faulty) / (drop + 1e-8)
            rec_str = f"{rec:+.1%}"

        print(f"  {label:<23} {mean:.4f} ± {ci:.4f}  "
              f"{d:>8.4f}  {rec_str:>10}")

    print("-"*65)
    print(f"\n95% CI = ±1.96·σ/√{n_seeds}")

    # Save
    df = pd.DataFrame(all_scores)
    df.to_csv("/kaggle/working/activation_fault_results.csv", index=False)

    summary = {
        "fault_type":     fault_type,
        "fault_fraction": fault_fraction,
        "n_seeds":        n_seeds,
        "n_problems":     n_problems,
        "scores":         all_scores,
    }
    with open("/kaggle/working/activation_fault_summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    print("Saved → /kaggle/working/activation_fault_results.csv")
    return all_scores


# ══════════════════════════════════════════════════════════════════════════════
# FAULT FRACTION SWEEP
# Find the fraction that gives consistent signal
# ══════════════════════════════════════════════════════════════════════════════

def sweep_fault_fraction(model, tokenizer, n_problems=50, n_seeds=3):
    """
    Quick sweep to find the right fault_fraction.
    Too small → no signal. Too large → model crashes.
    """
    from deepeval.benchmarks import IFEval

    print("Finding optimal fault fraction...")

    lfm   = LFM2(model=model, tokenizer=tokenizer)
    bench = IFEval(n_problems=n_problems)
    bench.evaluate(model=lfm)
    clean = bench.overall_score
    print(f"Clean: {clean:.4f}\n")

    for frac in [0.0001, 0.001, 0.005, 0.01, 0.05, 0.1]:
        scores = []
        for seed in range(n_seeds):
            random.seed(seed); torch.manual_seed(seed)
            handles = []
            for li in range(14):
                h = model.model.layers[li].register_forward_hook(
                    make_fault_hook("nan", frac)
                )
                handles.append(h)

            lfm   = LFM2(model=model, tokenizer=tokenizer)
            bench = IFEval(n_problems=n_problems)
            bench.evaluate(model=lfm)
            scores.append(bench.overall_score)

            for h in handles:
                h.remove()

        mean = np.mean(scores)
        drop = clean - mean
        print(f"  frac={frac:.4f}: {mean:.4f}  drop={drop:.4f} "
              f"({drop/clean*100:.1f}%)")

    return clean

In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════

print("="*65)
print("ACTIVATION FAULT INJECTION + LAAR EXPERIMENT")
print("LFM2.5-230M | IFEval | deepeval")
print("="*65)

# Step 1: Find right fault fraction (~5 min)
print("\n── Step 1: Fault fraction sweep ──")
clean_score = sweep_fault_fraction(
    model, tokenizer, n_problems=50, n_seeds=3
)

# Step 2: Full experiment at optimal fraction
# Based on sweep — use the fraction that gives 5-15% drop
# Typically 0.01 works well. Adjust based on sweep output.
print("\n── Step 2: Full experiment ──")
results = run_full_experiment(
    model          = model,
    tokenizer      = tokenizer,
    fault_type     = "nan",
    fault_fraction = 0.01,      # adjust after sweep
    target_layers  = list(range(14)),  # all layers
    n_problems     = 100,
    n_seeds        = 5,
)

ACTIVATION FAULT INJECTION + LAAR EXPERIMENT
LFM2.5-230M | IFEval | deepeval

── Step 1: Fault fraction sweep ──
Finding optimal fault fraction...


README.md: 0.00B [00:00, ?B/s]

ifeval_input_data.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/541 [00:00<?, ? examples/s]

Processing 50 IFEval problems: 100%|██████████| 50/50 [00:55<00:00,  1.11s/it]


...

Overall IFEval Accuracy: 0.6700
Instruction 'punctuation:no_comma' Accuracy: 0.4167
Instruction 'detectable_format:number_highlighted_sections' Accuracy: 0.8571
Instruction 'length_constraints:number_words' Accuracy: 0.2500
Instruction 'detectable_content:number_placeholders' Accuracy: 0.3333
Instruction 'combination:repeat_prompt' Accuracy: 1.0000
Instruction 'detectable_format:title' Accuracy: 1.0000
Instruction 'change_case:english_lowercase' Accuracy: 0.0000
Instruction 'detectable_format:number_bullet_lists' Accuracy: 1.0000
Instruction 'change_case:english_capital' Accuracy: 1.0000
Instruction 'detectable_format:multiple_sections' Accuracy: 1.0000
Instruction 'change_case:capital_word_frequency' Accuracy: 1.0000
Instruction 'startend:quotation' Accuracy: 1.0000
Instruction 'keywords:existence' Accuracy: 1.0000
Instruction 'detectable_format:json_format' Accuracy: 1.0000
Instruction 'length_constraints:number_paragraphs' Accuracy: 1.0000
Instruction 'combination:two_responses' Acc

Processing 100 IFEval problems: 100%|██████████| 100/100 [02:28<00:00,  1.48s/it]


Overall IFEval Accuracy: 0.6600
Instruction 'punctuation:no_comma' Accuracy: 0.4167
Instruction 'detectable_format:number_highlighted_sections' Accuracy: 0.7143
Instruction 'length_constraints:number_words' Accuracy: 0.2500
Instruction 'detectable_content:number_placeholders' Accuracy: 0.6667
Instruction 'combination:repeat_prompt' Accuracy: 1.0000
Instruction 'detectable_format:title' Accuracy: 1.0000
Instruction 'change_case:english_lowercase' Accuracy: 0.0000
Instruction 'detectable_format:number_bullet_lists' Accuracy: 1.0000
Instruction 'change_case:english_capital' Accuracy: 1.0000
Instruction 'detectable_format:multiple_sections' Accuracy: 1.0000
Instruction 'change_case:capital_word_frequency' Accuracy: 1.0000
Instruction 'startend:quotation' Accuracy: 1.0000
Instruction 'keywords:existence' Accuracy: 1.0000
Instruction 'detectable_format:json_format' Accuracy: 1.0000
Instruction 'length_constraints:number_paragraphs' Accuracy: 1.0000
Instruction 'combination:two_responses' Acc

Processing 100 IFEval problems: 100%|██████████| 100/100 [02:43<00:00,  1.64s/it]

Overall IFEval Accuracy: 0.6800
Instruction 'punctuation:no_comma' Accuracy: 0.4167
Instruction 'detectable_format:number_highlighted_sections' Accuracy: 0.8571
Instruction 'length_constraints:number_words' Accuracy: 0.2500
Instruction 'detectable_content:number_placeholders' Accuracy: 0.6667
Instruction 'combination:repeat_prompt' Accuracy: 1.0000
Instruction 'detectable_format:title' Accuracy: 1.0000
Instruction 'change_case:english_lowercase' Accuracy: 0.0000
Instruction 'detectable_format:number_bullet_lists' Accuracy: 1.0000
Instruction 'change_case:english_capital' Accuracy: 1.0000
Instruction 'detectable_format:multiple_sections' Accuracy: 1.0000
Instruction 'change_case:capital_word_frequency' Accuracy: 1.0000
Instruction 'startend:quotation' Accuracy: 1.0000
Instruction 'keywords:existence' Accuracy: 1.0000
Instruction 'detectable_format:json_format' Accuracy: 1.0000
Instruction 'length_constraints:number_paragraphs' Accuracy: 1.0000
Instruction 'combination:two_responses' Acc

In [5]:
# # What are the actual gate magnitudes across LIV layers?
# # We need to know if 0.1 is even the right threshold

# captured_mags = {}

# def make_mag_hook(li):
#     def hook(module, inp, out):
#         h = out[0] if isinstance(out, tuple) else out
#         valid = h[~torch.isnan(h) & ~torch.isinf(h)]
#         if len(valid) > 0:
#             captured_mags[li] = valid.abs().mean().item()
#     return hook

# # Run on clean model with one input
# handles = []
# for li in LIV_LAYERS:
#     h = model.model.layers[li].register_forward_hook(make_mag_hook(li))
#     handles.append(h)

# inputs = tokenizer(
#     "Please respond carefully to the following instruction:",
#     return_tensors="pt"
# ).to(device)
# with torch.no_grad():
#     _ = model(**inputs)

# for h in handles:
#     h.remove()

# print("Gate magnitudes per LIV layer:")
# for li, mag in sorted(captured_mags.items()):
#     print(f"  Layer {li:2d}: {mag:.4f}")
# print(f"\nMean: {np.mean(list(captured_mags.values())):.4f}")
# print(f"Your threshold was: 0.1")
# print(f"Correct threshold should be: "
#       f"{np.mean(list(captured_mags.values())):.4f}")